# Логистическая регрессия — мини-проект

Сквозной мини-проект: один датасет, решения передаются между этапами. См. docs/project_notebook.md.


In [ ]:
# Setup
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print
import warnings
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_score, recall_score

np.random.seed(42)



## Постановка задачи `(intro)`


Классифицируем **кредитных клиентов** (OpenML `credit-g`): «хороший» vs «плохой» заёмщик.

**Сюжет:** EDA → дисбаланс классов → ловушка Accuracy → `class_weight` → подбор `C` → Pipeline со scaling → ROC/PR/калибровка → порог → интерпретация — **один** датасет и **один** test.


## Загрузка и EDA `(eda)`


In [ ]:
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", font_scale=1.05)

credit = fetch_openml("credit-g", version=1, as_frame=True, parser="auto")
df = credit.frame.copy()
y = (df["class"] == "bad").astype(int)
X = pd.get_dummies(df.drop(columns=["class"]), drop_first=True)

print(f"Объектов: {len(df)}, признаков после OHE: {X.shape[1]}")
print(f"Доля класса 'bad': {y.mean():.3f}")

fig, ax = plt.subplots(figsize=(4, 3))
y.value_counts().sort_index().plot(kind="bar", ax=ax, color=["steelblue", "crimson"])
ax.set_xticklabels(["good (0)", "bad (1)"], rotation=0)
ax.set_title("Дисбаланс классов")
ax.set_ylabel("число объектов")
plt.tight_layout()
plt.show()


## Решение: train / test split `(example)`


**Решение:** стратифицированный split (`random_state=42`). Этот test — до конца ноутбука.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, test: {len(X_test)}")
print("Доля bad в train:", y_train.mean().round(3), "| test:", y_test.mean().round(3))


## Ловушка Accuracy `(experiment)`


Модель «всегда good» даёт высокий accuracy, но бесполезна для поиска «плохих» клиентов.


In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
print(f"Dummy accuracy: {dummy.score(X_test, y_test):.3f}\n")

pipe_baseline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42),
)
pipe_baseline.fit(X_train, y_train)
print("LogReg без class_weight:")
print(classification_report(y_test, pipe_baseline.predict(X_test), target_names=["good", "bad"], digits=3))


## Решение: class_weight='balanced' `(example)`


**Решение:** включаем `class_weight='balanced'` — усиливаем штраф за ошибки на минорном классе «bad».


In [ ]:
USE_CLASS_WEIGHT = "balanced"

pipe_balanced = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight=USE_CLASS_WEIGHT, max_iter=2000, random_state=42),
)
pipe_balanced.fit(X_train, y_train)
print("LogReg с class_weight='balanced':")
print(classification_report(y_test, pipe_balanced.predict(X_test), target_names=["good", "bad"], digits=3))


## Подбор регуляризации C `(experiment)`


Сравниваем `C` на **том же** train/test с уже выбранным `class_weight`.


In [ ]:
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
rows = []
for C in C_values:
    m = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C, class_weight=USE_CLASS_WEIGHT, max_iter=2000, random_state=42),
    )
    m.fit(X_train, y_train)
    proba = m.predict_proba(X_test)[:, 1]
    rows.append({
        "C": C,
        "accuracy": m.score(X_test, y_test),
        "PR-AUC": average_precision_score(y_test, proba),
        "norm_w": float(np.linalg.norm(m.named_steps["logisticregression"].coef_)),
    })

c_df = pd.DataFrame(rows).sort_values("PR-AUC", ascending=False)
display(c_df.round(4))


## Решение: финальная модель `(example)`


**Решение:** берём `C` с лучшим PR-AUC на test. Обучаем `final_pipe` — дальше только она.


In [ ]:
BEST_C = float(c_df.iloc[0]["C"])

final_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=BEST_C, class_weight=USE_CLASS_WEIGHT, max_iter=2000, random_state=42),
)
final_pipe.fit(X_train, y_train)
proba = final_pipe.predict_proba(X_test)[:, 1]
pred = final_pipe.predict(X_test)

print(f"Финальная модель: C={BEST_C}, class_weight={USE_CLASS_WEIGHT!r}")
print(classification_report(y_test, pred, target_names=["good", "bad"], digits=3))


## ROC, PR и калибровка `(viz)`


При дисбалансе смотрим **PR-кривую**. Reliability diagram проверяет калибровку вероятностей.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0])
axes[0].set_title("ROC")

PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[1])
axes[1].set_title("PR (дисбаланс)")

frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=8, strategy="uniform")
axes[2].plot([0, 1], [0, 1], "k--", label="идеал")
axes[2].plot(mean_pred, frac_pos, "o-", color="crimson", label="final_pipe")
axes[2].set_xlabel("средняя P(bad)")
axes[2].set_ylabel("доля bad")
axes[2].set_title("Калибровка")
axes[2].legend()
plt.tight_layout()
plt.show()


## Выбор порога `(viz)`


Порог 0.5 не обязан быть оптимальным. Сравним precision/recall при разных порогах.


In [ ]:
rows_thr = []
for thr in [0.5, 0.35, 0.25]:
    y_p = (proba >= thr).astype(int)
    rows_thr.append({
        "threshold": thr,
        "precision": precision_score(y_test, y_p, zero_division=0),
        "recall": recall_score(y_test, y_p, zero_division=0),
    })
thr_df = pd.DataFrame(rows_thr)
display(thr_df.round(3))

CHOSEN_THRESHOLD = 0.35
print(f"\n**Решение:** для баланса recall/precision берём порог {CHOSEN_THRESHOLD}")


## Интерпретация: log-odds `(example)`


Вес $w_j$ — изменение log-odds при +1 sigma признака после scaler (при прочих равных).


In [ ]:
lr = final_pipe.named_steps["logisticregression"]
w = lr.coef_.ravel()
feat_names = X.columns
top_idx = np.argsort(np.abs(w))[-8:]

plt.figure(figsize=(8, 4))
plt.barh(np.array(feat_names)[top_idx], w[top_idx], color="steelblue")
plt.xlabel("вес (log-odds)")
plt.title("Топ-8 признаков |w| (финальная модель)")
plt.tight_layout()
plt.show()

j = top_idx[-1]
print(f"Признак: {feat_names[j]}")
print(f"  w = {w[j]:+.3f} → odds × {np.exp(w[j]):.2f} при +1 sigma")


## Чек-лист мини-проекта `(summary)`


1. Один датасет `credit-g`, один stratified split.
2. При дисбалансе — не верить Accuracy; `class_weight='balanced'`.
3. Подбор `C` → **фиксируем** `final_pipe`.
4. PR-кривая важнее ROC; калибровка на hold-out.
5. Порог — бизнес-решение, не обязательно 0.5.
6. Интерпретация через log-odds на **финальной** модели.
